# Topical Lectures, Controls
## Real drone, part 5: tuning the position loop

Andreas Freise, 31.10.2026

### Goal

In the morning you tuned the loops of the simulated drone: a proportional gain alone let the tilt oscillate, a damping term calmed it, an integrator removed the offset in the altitude. On the simulator that costs nothing. A run takes a second, the same gains give the same curve every time, and nothing breaks. Now the same loop runs on the real drone. The gains do the same things, but a flight with preparation takes minutes, no two flights are alike, the estimate is noisy, the feedback acts with a delay, and a setting that oscillates has to be aborted by hand. This notebook is for a tuning session: we set one gain at a time, using a prediction before each flight.

Every flight is the same: the drone takes off under the firmware, hovers, hands `x` and `y` to our controller, holds for 2 s, the target jumps by `STEP`, 4 s hold, landing. The controller is the one of the hand test, defined again in the first code cell below; the parameters of your drone come from the identification flight. Every gain is entered by you, in the cell where each flight starts.

### Getting close without crashing

* Get a prediction for every flight before you fly it, from the model drone of your group, and aim at a rise time of two to three seconds.
* Keep `|Kp|` times the step below `v_max`: beyond that the first part of the response is a flight at the limit, the same for all gains.
* Change one gain per flight, and double it rather than jumping tenfold.
* Stop raising `Kp` when the overshoot passes 10 cm: the dead time sets the ceiling.
* The fence lands the drone when it leaves the 1 m box around the start, and Enter lands it at any time. An aborted flight is a measurement too: save it.

### Tools for the rest of the group

The drone laptop is busy with the flights. The model of the drone runs on the other laptops of the group in `drone_analysis_01_flight_plan`, from the parameters copied in below: it gives the first gain to fly and, before each flight, the predicted rise time and overshoot for the gains about to be tried. Once there are flights to show, a copy of the `data/` folder of this laptop goes to `drone_analysis_02_summary_figure` for the table and the figure of the summary.

### Safety

* Fly only in the marked area. One person at the keyboard, one watching the drone, nobody within arm's length while the motors run.
* The person at the keyboard keeps the abort window in front, clicked once so that it takes the keys. **Enter** aborts and lands. The stop button of the cell also lands; press it once only. The window closes by itself after the landing.
* The loop lands by itself when the drone leaves the 1 m box around the start, when no state arrives any more, when the battery drops below 3.1 V under load, or when the position estimate becomes unreliable.
* Full battery for the tuning runs: below 3.5 V at rest the drone lacks thrust and the runs cannot be compared. After a tumble or crash, run the status cell; if it says locked, the cell after it clears the lock.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = None
URI = crazyflie.drone_uri(LABEL)

## The results so far

Copy your recorded values into the next cell: the sign of the gains, the response time and the dead time, the two noise bounds. `STEP` is the size of the step in every flight, and `FLIGHTS` is your record of the flights, filled after each one.


In [ ]:
# Copy your recorded results here:
KP_SIGN = None   # +1 or -1, the sign of the gains (hand test)
TAU = None       # s, the response time of the velocity loop (identification flight)
DELAY = None     # s, the dead time (identification flight)
KP_MAX = None    # 1/s, the bound on |Kp| from the noise (identification flight)
KD_MAX = None    # the bound on |Kd| from the noise (identification flight)

STEP = 0.5       # m, the step of the target in every flight
FLIGHTS = []     # starts empty; the record cell after each flight below appends one entry


## The code

The controller of the hand test once more, and the lines that read a step response: rise time, overshoot and the error in the last two seconds.


In [ ]:
dt = 0.02        # s, the loop runs at 50 Hz
v_max = 0.3      # m/s, the limit on the feedback, a velocity

def make_controller(Kp, Kd, Ki):
    """One PID per axis with these gains. Returns the controller that fly() calls and the reset of its integrators."""
    x_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)
    y_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)

    def position_controller(x, y, vx, vy, x_target, y_target):
        """Position and velocity in, feedback (a velocity) out, clipped to v_max."""
        vx_fb = x_ctrl(x, vx, x_target)
        vy_fb = y_ctrl(y, vy, y_target)
        return float(np.clip(vx_fb, -v_max, v_max)), float(np.clip(vy_fb, -v_max, v_max))

    def reset_controllers():
        """Zero the integrators, called before every flight."""
        x_ctrl.reset()
        y_ctrl.reset()

    return position_controller, reset_controllers


In [ ]:
def step_numbers(t, x, step, fb=None, v_max=None):
    """Read one step response: t in seconds from the step, x in metres from the start, step the change of the target."""
    after = t >= 0
    t, x, step = t[after], x[after] * np.sign(step), abs(step)
    hit = x > 0.9 * step
    rise = float(t[np.argmax(hit)]) if hit.any() else None          # first time beyond 90 % of the step
    overshoot = float(max(0.0, x.max() - step))                     # the largest excursion beyond the target
    tail = t >= t[-1] - 2.0                                         # the last 2 s of the step phase
    err = step - x[tail]
    numbers = {'rise_s': rise, 'overshoot_m': overshoot, 'max_error_m': float(np.abs(err).max()),
               'mean_error_m': float(err.mean()), 'settled': bool(np.abs(err).max() < 0.15)}
    if fb is not None and v_max is not None:
        numbers['clamp_fraction'] = float((np.abs(fb[after]) >= v_max - 1e-3).mean())   # share of the step at the limit
    return numbers

def print_numbers(*columns):
    """A small table of step numbers: print_numbers(('model', pred), ('flight', meas))."""
    show = lambda v: '--' if v is None else ('yes' if v is True else 'no' if v is False else f'{v:.2f}')
    print(f"{'':26s}" + ''.join(f'{name:>12s}' for name, _ in columns))
    for key, text in (('rise_s', 'rise to 90 % [s]'), ('overshoot_m', 'overshoot [m]'), ('max_error_m', 'max error, last 2 s [m]'),
                      ('mean_error_m', 'mean error, last 2 s [m]'), ('clamp_fraction', 'share at v_max'), ('settled', 'settled (< 0.15 m)')):
        if any(key in n for _, n in columns):
            print(f'{text:26s}' + ''.join(f'{show(n.get(key)):>12s}' for _, n in columns))


## Test of the abort

Once per session: the cell runs a dummy loop for 10 s with the same abort window as the flights. Click into the window and press Enter; the loop must report the abort. If it does not, do not fly from this program.


In [ ]:
crazyflie.bench_test_abort()

## Status before take-off

Run this before every flight: battery above 3.5 V, flow deck attached, supervisor not locked. `is locked: yes` means the supervisor has locked the motors after a tumble, a crash or a hard landing; the cell after this one clears the lock and shows the status again. If it stays locked, switch the drone off and on.


In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)

In [ ]:
# Only when the status says locked: clear the lock, then check again.
with crazyflie.connect(URI) as scf:
    crazyflie.recover(scf)
    crazyflie.status(scf)

## The gains for the next flight

Every flight starts here. Enter the three gains and run the cell: it checks the feedback at the moment of the step against `v_max` and the gains against the noise bounds, and builds the controller. Before you fly, write down what you expect, the rise time and the overshoot; the model drone of your group gives numbers for it.

For Flight 2 choose `Kp` too small on purpose: the sign the hand test found and a size between 0.05 and 0.2, with `Kd = 0.0` and `Ki = 0.0`. The drone will not reach the target within the step. Fly it anyway: the first flight under your own loop should be one in which nothing can go wrong. For Flight 3 take the gain the model drone suggests; without one, `KP_SIGN` times 1.0 is a reasonable start on these drones. From then on you choose: a rise time of two to three seconds, with the feedback at the step below `v_max`.


In [ ]:
Kp = None     # 1/s: feedback (a velocity) per metre of position error; the sign from the hand test
Kd = None     # feedback per m/s of measured velocity; with the sign of Kp it damps, with the opposite sign it removes damping
Ki = None     # 1/s^2: feedback per metre second of accumulated error
if None in (Kp, Kd, Ki):
    raise ValueError('enter the three gains first; the text above says where to start')
if np.sign(Kp) != KP_SIGN:
    print('WARNING: the sign of Kp is not the one the hand test found')

at_step = abs(Kp) * STEP                           # the feedback at the moment of the step
print(f'feedback at the step {at_step:.2f} m/s, v_max {v_max:.2f} m/s: '
      + ('at the limit, the first part of the response is a flight at v_max' if at_step > v_max else 'below the limit'))
exceeded = abs(Kp) > KP_MAX or abs(Kd) > KD_MAX     # the gains against the noise bounds
print(f'noise bounds: |Kp| {abs(Kp):.2f} of {KP_MAX:.1f}, |Kd| {abs(Kd):.2f} of {KD_MAX:.1f}' + ('   EXCEEDED' if exceeded else ''))

RUN_INFO = dict(Kp=Kp, Kd=Kd, Ki=Ki, tau=TAU, delay=DELAY)   # saved with the flight, for the comparison with the model later
position_controller, reset_controllers = make_controller(Kp, Kd, Ki)   # the code that flies, with these gains


## Flight 2: the first step, with a gain too small on purpose

The loop closes on the real drone for the first time. Put the drone flat on the floor, with the forward side into the free space: the step goes in that direction. Announce the flight and run the cell. The drone takes off under the firmware, hands over to our controller, holds for 2 s, the target steps, it holds for another 4 s and lands; the printout shows the position at the handover and at the step. If the drone drifts away or oscillates visibly, press Enter in the abort window and it lands.

`RUN_LABEL` names the run: `kp_small` for this one, `kp1` for `|Kp| = 1`, `kp2_kd03` later, `kp1_again` for a repeat. The cell saves the run as `flight_<drone label>_<time>_<RUN_LABEL>_*.csv` in `data/`, the gains with it, and prints its rise time, overshoot and settling error.


In [ ]:
RUN_LABEL = 'kp_small'            # what this run tries; goes into the file names, change it for every run
AXES = 'x'                       # 'x', 'y' or 'xy', the axes your controller acts on

run = crazyflie.fly(position_controller, axes=AXES, step=STEP, v_max=v_max,
                    reset=reset_controllers, uri=URI, info=RUN_INFO)
name = crazyflie.save(run, RUN_LABEL)


## The step response of the flight

The saved run is loaded and drawn: above the position along the stepped axis, with the 15 cm band around the target; below the feedback against the measured velocity. Under the plot the numbers of the flight. Compare them with the prediction you wrote down: which number differs most, and what explains it, the dead time, the limit of the feedback, the noise, or something the model leaves out?


In [ ]:
r = crazyflie.load_run(name)
if r['t_step'] is None:
    print('this run was aborted before the step, nothing to plot')
else:
    pid = r['pid']
    t = pid['t_s'] - r['t_step']
    g = r['info']                     # the gains this run was flown with
    fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    for a in r['axes']:
        ax[0].plot(t, pid[a] - pid[f'target_{a}'][0], label=f"{a} position, Kp = {g['Kp']:+.2f}, Kd = {g['Kd']:+.2f}, Ki = {g['Ki']:+.2f}")
        line, = ax[1].plot(t, pid[f'fb_{a}'], label=f'{a} feedback')
        ax[1].plot(t, pid[f'v{a}'], color=line.get_color(), alpha=0.5, label=f'{a} measured velocity')
    ax[0].axhspan(g['step'] - 0.15, g['step'] + 0.15, color='0.92', zorder=0)
    ax[0].axhline(g['step'], color='k', lw=0.5, ls='--')
    ax[0].axhline(0, color='k', lw=0.5, ls='--')
    ax[0].set_ylabel('position from start [m]')
    ax[0].legend(loc='lower right')
    ax[0].grid()
    ax[1].axhline(g['v_max'], color='k', lw=0.5, ls='--')
    ax[1].axhline(-g['v_max'], color='k', lw=0.5, ls='--')
    ax[1].set_ylabel('velocity [m/s]')
    ax[1].set_xlabel('time since step [s]')
    ax[1].legend(loc='upper right')
    ax[1].grid()
    plt.tight_layout()

    a = r['axes'][0]
    meas = step_numbers(t, pid[a] - pid[f'target_{a}'][0], pid[f'target_{a}'][-1] - pid[f'target_{a}'][0], pid[f'fb_{a}'], g['v_max'])
    print_numbers(('flight', meas))


### Record this flight

Add one entry to `FLIGHTS` from the numbers above: the label and the gains are filled in; the prediction you wrote down before the flight, the measured rise time and overshoot, whether it settled and a note in your own words are yours. This record, next to the table, is what you show in the summary.


In [ ]:
FLIGHTS.append(dict(label=RUN_LABEL, Kp=Kp, Kd=Kd, Ki=Ki,
                    predicted_rise_s=None,   # s, the prediction you wrote down before the flight
                    rise_s=None,          # s, from the printout above
                    overshoot_m=None,     # m
                    settled=None,         # True or False
                    note=''))             # what you saw, in a few words


### Milestone 4: the loop is closed, and too slow

The numbers of this run say "settled: no", and the plot shows the drone on its way to the target but not there within the step. That is what we expect: the loop is closed with the right sign, and the gain is too small. The milestone is passed when that is what you see. Had the drone moved away from the target, the sign would have been wrong; Enter lands the drone, and the fence does the same at 1 m.


## The table

One line per saved flight of your drone, the gains and the numbers of its step. Run it after every flight; the table is where you see which gain changed what.


In [ ]:
mine = crazyflie.load_runs('data', drone=LABEL)
rows = crazyflie.run_table(mine)

## Flight 3: the first real gain

Now the loop should settle for the first time. Go back to the gains cell and enter the gain the model drone of your group suggests, or `KP_SIGN` times 1.0, with `Kd` and `Ki` at 0.0. Run the cells again down to the table, with a new `RUN_LABEL`.

### Milestone 5: the drone flies under our loop

The milestone is passed when the numbers of the run say "settled: yes": the drone arrived at the target within 15 cm and stayed there for the last 2 s of the step. Compare the measured rise time with the prediction; the difference is the first thing about the live system that the model did not know.

## Flights 4 and onwards: one gain at a time

Every further flight uses the same cells: the gains, the flight with a new `RUN_LABEL`, the comparison, the record, the table. Change one gain per flight and write down the prediction before you fly. The first two series fill the afternoon; the rest is for groups with time left.

1. **`Kp`, the speed.** Double it from the value of Flight 3, then again, as long as the drone settles. Get the prediction for each flight before you fly it, and note where the flight before departed from its prediction. Watch three things: the rise time, which stops shrinking once the feedback sits at `v_max`; the overshoot, which the dead time adds; and the feedback in the lower panel, which starts to swing. The largest `Kp` that still settles is a number to remember, and the dead time is what sets it.
2. **`Kd`, the damping.** At the `Kp` where the overshoot appeared, fly `Kd` at 0.3 and then at 0.6, with the sign of `Kp`. Does the overshoot go? Compare the feedback in the lower panel with the flight without `Kd`: the damping term multiplies the velocity estimate, noise and all. Then one flight with `Kd` of the opposite sign, because the sign matters.
3. **`Ki`, the offset.** Look at the mean error in the last 2 s of your flights so far. With a proportional gain alone the drone settles where the feedback balances the drift of the estimate and the air in the room, not at the target. Fly `|Kp| = 1` with `Ki` at 0.1, both with the sign of `Kp`. Does the mean error go? What happened to the overshoot, and why?
4. **The same gains twice.** Fly the setting of Flight 3 again at the end and compare the two rows. No tuning can be finer than the difference between two flights with the same gains; a model drone gives the same answer every time, apart from its noise.
With time left: a small step (`STEP = 0.15` in the results cell) with a large `Kp`, where the feedback stays below `v_max`, and both axes at once (`AXES = 'xy'`).

## Questions for the summary

Look at your record, your table and the plots of the runs behind them, and answer for the summary:

* Which gain changed the rise time, which the overshoot, which the mean error? Where did a larger gain change nothing, and what stood in the way, the limit of the feedback or the dead time?
* Where did the flight differ from the model, and which of the three, dead time, feedback limit, noise, or what the model leaves out, the air in the room and the attitude loop, explains it?
* Two flights with the same gains: how different were they, and what does that mean for how finely one can tune a live system?
* How many flights did your best setting take, and how many would it have taken on a model that gives the same answer every time?

Then record your best gains in the cell below. The figure for the summary is made from the `data/` folder of this laptop with `drone_analysis_02_summary_figure`.


In [ ]:
# Record your result of the afternoon:
BEST_GAINS = dict(Kp=None, Kd=None, Ki=None)   # the setting you would fly again, and why (write it in the note of its flight)


## If something goes wrong

If a flight fails, these are the messages and symptoms seen so far, and what to do about each.

* `No drone answers at ...`: the drone is off, or not on the channel of its label.
* `is locked: yes` after a tumble, a crash or a hard landing: run the recover cell under the status cell; if it stays locked, switch the drone off and on.
* `No flow deck detected`: switch off, reseat the deck, switch on.
* The drone wanders off during the hold although the feedback is small: the floor has too little texture for the flow sensor. Fly over the mat, or check what the sensor sees while you move the drone by hand over the floor: `with crazyflie.connect(URI) as scf: crazyflie.flow_check(scf)` in a cell.
* `state samples stale`: the radio link dropped. Check the Crazyradio and the address, keep the laptop close to the flying area.
* The loop never prints `handover`: the take-off failed, usually a low battery or a locked supervisor.
* `a flow deck that was not delivering`, `does not follow the height sensor`, `runs away while the drone lies still`, `after the take-off` or `height sensor frozen`: the deck stopped delivering, which happens without any other sign, also right after switching on. Switch the drone off and on, connect again.
